In [ ]:
pip install yfinance

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm


def obtener_variacion_logaritmica_porcentaje(ticker):
    """
    Esta función devuelve un DataFrame con las variaciones logarítmicas de los precios de cierre
    de un ticker dado para un rango de fechas, expresadas en porcentaje y con el formato decimal ajustado
    para usar comas como separadores decimales, además de añadir el símbolo de porcentaje.

    Parámetros:
    - ticker: El símbolo del ticker de la acción (como string).
    - fecha_inicio: La fecha de inicio del rango en formato 'AAAA-MM-DD'.
    - fecha_fin: La fecha de fin del rango en formato 'AAAA-MM-DD'.
    """
    fecha_inicio = "2018-01-01"
    fecha_fin = "2023-12-31"
    # Descargar los datos del ticker
    datos = yf.download(ticker, start=fecha_inicio, end=fecha_fin)

    # Seleccionar solo la columna 'Close'
    precios_cierre = datos[['Close']]

    # Calcular la variación logarítmica de los precios de cierre
    variacion_log = np.log(precios_cierre / precios_cierre.shift(1))

    # Convertir la variación logarítmica a formato porcentual
    variacion_log_porcentaje = variacion_log * 100

    # Convertir a string, usar coma como separador decimal y añadir el símbolo de porcentaje
    variacion_log_porcentaje = variacion_log_porcentaje['Close'].replace('.', ',')

    # Crear un nuevo DataFrame para devolver, usando la fecha como índice
    df_resultado = pd.DataFrame(variacion_log_porcentaje)
    df_resultado.rename(columns={'Close': 'Variacion Logaritmica (%)'}, inplace=True)

    return df_resultado



In [19]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm

def regresion(ticker):
    # Suponiendo que tienes una función obtener_variacion_logaritmica_porcentaje que recoge los datos y realiza el cálculo
    df_variacion_log = obtener_variacion_logaritmica_porcentaje(ticker)

    # Cargando un DataFrame externo que supongamos contiene datos como 'Mkt-RF', 'SMB', 'HML', etc.
    famafrench_df = pd.read_csv('/content/csv_general_dailychange.csv', sep=';')
    famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'])
    famafrench_df.set_index('Date', inplace=True)

    # Combinar los DataFrames
    df_combinado = famafrench_df.merge(df_variacion_log, left_index=True, right_index=True, how='outer')
    df_combinado.dropna(inplace=True)

    df_combinado.index = pd.to_datetime(df_combinado.index)

    # Define el rango de fechas
    fecha_inicio = '2018-01-01'
    fecha_fin = '2023-12-31'
    print(df_combinado)
    # Filtra el DataFrame para incluir solo las fechas dentro del rango
    df_combinado = df_combinado[(df_combinado.index >= fecha_inicio) & (df_combinado.index <= fecha_fin)]

    df_combinado.round(3)
    print(df_combinado)
    # Ajustando y formateando los datos como se necesita
    df_combinado['Mkt-RF'] = df_combinado['Mkt-RF'].astype(str).str.replace(',', '.').astype(float)
    df_combinado['SMB'] = df_combinado['SMB'].astype(str).str.replace(',', '.').astype(float)
    df_combinado['HML'] = df_combinado['HML'].astype(str).str.replace(',', '.').astype(float)
    df_combinado['RF'] = df_combinado['RF'].replace(',', '.').astype(float) / 100
    df_combinado['Variacion Logaritmica (%)'] = df_combinado['Variacion Logaritmica (%)'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100
    df_combinado['Fundflows'] = df_combinado['Fundflows'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float)
    print(df_combinado)
    # Preparando variables para la regresión
    X = df_combinado[['Mkt-RF', 'SMB', 'HML', 'Fundflows']]
    X = sm.add_constant(X)
    Y = df_combinado['Variacion Logaritmica (%)'] - df_combinado['RF']

    # Ajustar modelo OLS
    modelo = sm.OLS(Y, X).fit()
    # Crear un DataFrame para este ticker específico con los resultados del modelo
    resultados_ticker = pd.DataFrame({
        ticker: {
            'const': modelo.params['const'],
            'pvalorconst': modelo.pvalues['const'],
            'coefSML': modelo.params['SMB'],
            'pvalorSML': modelo.pvalues['SMB'],
            'coefHML': modelo.params['HML'],
            'pvalorHML': modelo.pvalues['HML'],
            'r2': modelo.rsquared,
            'coefrmrf': modelo.params['Mkt-RF'],
            'pvalorrmrf': modelo.pvalues['Mkt-RF'],
            'coeffundflows': modelo.params['Fundflows'],
            'pvalorfundflows': modelo.pvalues['Fundflows']

        }
    })
    return resultados_ticker




In [20]:
# Asume que esta es tu lista de tickers
lista_tickers = ["MSFT", "AAPL", "NVDA", "AMZN", "META", "GOOGL", "GOOG", "BRK.B", "LLY", "AVGO", "JPM", "TSLA", "XOM", "V", "UNH", "MA", "PG", "JNJ", "HD", "MRK", "COST", "ABBV", "CRM", "CVX", "AMD", "NFLX", "BAC", "WMT", "PEP", "KO", "LIN", "TMO", "ADBE", "DIS", "ACN", "WFC", "ORCL", "CSCO", "MCD", "QCOM", "ABT", "CAT", "INTU", "AMAT", "IBM", "VZ", "GE", "CMCSA", "NOW", "INTC", "DHR", "COP", "UBER", "TXN", "PFE", "UNP", "AMGN", "PM", "LOW", "SPGI", "ISRG", "MU", "RTX", "GS", "NEE", "HON", "ETN", "AXP", "LRCX", "BKNG", "PGR", "T", "ELV", "SYK", "C", "MS", "PLD", "BLK", "MDT", "TJX", "NKE", "UPS", "SCHW", "DE", "CI", "BA", "VRTX", "BMY", "CB", "ADP", "MMC", "BSX", "REGN", "SBUX", "ADI", "LMT", "FI", "KLAC", "CVS", "BX", "MDLZ", "AMT", "SNPS", "GILD", "PANW", "CDNS", "TMUS", "CMG", "MPC", "EOG", "ICE", "TGT", "SHW", "SLB", "CME", "SO", "ZTS", "WM", "ANET", "DUK", "MO", "EQIX", "PH", "PSX", "CL", "ITW", "FCX", "PYPL", "CSX", "BDX", "MCK", "ABNB", "APH", "TT", "TDG", "USB", "GD", "ORLY", "EMR", "HCA", "NOC", "PNC", "PCAR", "AON", "FDX", "PXD", "NXPI", "MAR", "MCO", "VLO", "CEG", "CTAS", "MSI", "ROP", "ECL", "NSC", "EW", "COF", "AIG", "DXCM", "HLT", "AZO", "APD", "F", "TRV", "AJG", "ADSK", "TFC", "GM", "WELL", "MMM", "NUE", "SPG", "CPRT", "CARR", "MCHP", "URI", "ROST", "WMB", "DHI", "SMCI", "OKE", "PSA", "NEM", "OXY", "MET", "AFL", "ALL", "TEL", "GWW", "SRE", "O", "AEP", "IQV", "JCI", "AMP", "FTNT", "CCI", "MSCI", "DLR", "FAST", "FIS", "BK", "HES", "STZ", "IDXX", "KMB", "A", "DOW", "AME", "PRU", "LULU", "LEN", "MNST", "CMI", "D", "CTVA", "ODFL", "OTIS", "COR", "PAYX", "LHX", "GIS", "HUM", "CNC", "SYY", "RSG", "MLM", "CSGP", "PWR", "IR", "YUM", "EXC", "GEHC", "FANG", "IT", "HAL", "KR", "PCG", "VMC", "CTSH", "KMI", "GEV", "ACGL", "MRNA", "KVUE", "DG", "BKR", "DVN", "CDW", "EL", "ADM", "GPN", "PEG", "PPG", "VRSK", "DD", "RCL", "MPWR", "ROK", "KDP", "EA", "EFX", "EXR", "DFS", "ED", "HIG", "VICI", "FICO", "XYL", "DAL", "ANSS", "XEL", "BIIB", "FTV", "ON", "KHC", "HSY", "WST", "CBRE", "MTD", "KEYS", "WTW", "RMD", "EIX", "CHTR", "TSCO", "CAH", "WAB", "EBAY", "DLTR", "ZBH", "LYB", "TROW", "AVB", "HWM", "TRGP", "WEC", "HPQ", "WY", "NVR", "CHD", "PHM", "BLDR", "FITB", "DOV", "GLW", "RJF", "TTWO", "BR", "NDAQ", "STT", "WDC", "MTB", "HPE", "AWK", "IRM", "SBAC", "GRMN", "ALGN", "DECK", "DTE", "STLD", "ETR", "HUBB", "ULTA", "PTC", "MOH", "CPAY", "NTAP", "AXON", "EQR", "IFF", "APTV", "BAX", "GPC", "CTRA", "STE", "BALL", "ES", "ILMN", "INVH", "BRO", "PPL", "HBAN", "WAT", "FE", "ARE", "COO", "TDY", "LVS", "CBOE", "VLTO", "FSLR", "CINF", "AEE", "TXT", "MKC", "RF", "WBD", "DRI", "PFG", "J", "OMC", "NTRS", "HOLX", "IEX", "CLX", "CNP", "LH", "JBL", "WRB", "LDOS", "AVY", "EXPE", "SYF", "DPZ", "TYL", "VTR", "MAS", "ATO", "CMS", "MRO", "STX", "EXPD", "PKG", "LUV", "TSN", "FDS", "NRG", "SWKS", "VRSN", "TER", "EG", "CE", "CFG", "AKAM", "JBHT", "CCL", "ENPH", "ESS", "BBY", "SNA", "TRMB", "ALB", "BG", "EPAM", "MAA", "POOL", "CF", "ZBRA", "K", "EQT", "CAG", "SWK", "NDSN", "LYV", "DGX", "HST", "KEY", "UAL", "VTRS", "L", "LKQ", "WBA", "PNR", "DOC", "IP", "AMCR", "KMX", "RVTY", "CRL", "MGM", "ROL", "GEN", "JKHY", "WRK", "LNT", "KIM", "TAP", "AES", "EVRG", "IPG", "EMN", "SJM", "PODD", "JNPR", "ALLE", "FFIV", "HII", "UDR", "LW", "QRVO", "NI", "CPT", "TECH", "APA", "AOS", "BBWI", "MOS", "UHS", "CTLT", "INCY", "TFX", "WYNN", "HRL", "TPR", "PAYC", "NWSA", "REG", "DAY", "AIZ", "HSIC", "SOLV", "MTCH", "GL", "BF.B", "CZR", "AAL", "BXP", "CPB", "MKTX", "CHRW", "PNW", "GNRC", "BWA", "NCLH", "RHI", "ETSY", "FOXA", "BEN", "IVZ", "FMC", "FRT", "HAS", "DVA", "CMA", "BIO", "RL", "MHK",]

df_resultados_finales = pd.DataFrame()

# Procesar cada ticker y acumular los resultados
for ticker in lista_tickers:
    try:
        resultados_ticker = regresion(ticker)
        df_resultados_finales = pd.concat([df_resultados_finales, resultados_ticker], axis=1)
    except Exception as e:
        print(f"Error al procesar {ticker}: {e}")

# Transponer el DataFrame para tener tickers como índices y resultados como columnas
df_resultados_finales = df_resultados_finales.T

# Asegurarse de que los números están en el formato correcto, en este caso, separador decimal como punto
df_resultados_finales = df_resultados_finales.applymap(lambda x: float(str(x).replace(',', '.')))

# Exportar el DataFrame a CSV
df_resultados_finales.to_csv('/content/resultados_modelos.csv')

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed

           Mkt-RF    SMB    HML     RF     Fundflows  \
Date                                                   
2018-01-05   0,66  -0,35  -0,26  0,60%  -0,469042694   
2018-01-08   0,19  -0,15   0,05  0,60%   1,374904696   
2018-01-09   0,15  -0,35  -0,04  0,60%   -2,78345551   
2018-01-10  -0,07   0,07   0,57  0,60%  -0,468408524   
2018-01-11   0,87   1,15   0,31  0,60%   1,960887675   
...           ...    ...    ...    ...           ...   
2023-12-21   0,34  -0,43  -0,88  2,10%    0,12872561   
2023-12-22   1,55   1,72   1,35  2,10%   0,784379335   
2023-12-26   0,51   1,12   2,05  2,10%   0,042286528   
2023-12-27  -0,06   -0,4  -0,57  2,10%  -0,616815481   
2023-12-28   0,42  -0,38  -0,38  2,10%   0,614868291   

            Variacion Logaritmica (%)  
Date                                   
2018-01-05                   1.232191  
2018-01-08                   0.101996  
2018-01-09                  -0.067986  
2018-01-10                  -0.454445  
2018-01-11                   0.


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed

           Mkt-RF    SMB    HML     RF     Fundflows  \
Date                                                   
2018-01-05   0,66  -0,35  -0,26  0,60%  -0,469042694   
2018-01-08   0,19  -0,15   0,05  0,60%   1,374904696   
2018-01-09   0,15  -0,35  -0,04  0,60%   -2,78345551   
2018-01-10  -0,07   0,07   0,57  0,60%  -0,468408524   
2018-01-11   0,87   1,15   0,31  0,60%   1,960887675   
...           ...    ...    ...    ...           ...   
2023-12-21   0,34  -0,43  -0,88  2,10%    0,12872561   
2023-12-22   1,55   1,72   1,35  2,10%   0,784379335   
2023-12-26   0,51   1,12   2,05  2,10%   0,042286528   
2023-12-27  -0,06   -0,4  -0,57  2,10%  -0,616815481   
2023-12-28   0,42  -0,38  -0,38  2,10%   0,614868291   

            Variacion Logaritmica (%)  
Date                                   
2018-01-05                   0.843846  
2018-01-08                   3.018065  
2018-01-09                  -0.027030  
2018-01-10                   0.780934  
2018-01-11                   0.


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed

...

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


           Mkt-RF    SMB    HML     RF     Fundflows  \
Date                                                   
2018-01-05   0,66  -0,35  -0,26  0,60%  -0,469042694   
2018-01-08   0,19  -0,15   0,05  0,60%   1,374904696   
2018-01-09   0,15  -0,35  -0,04  0,60%   -2,78345551   
2018-01-10  -0,07   0,07   0,57  0,60%  -0,468408524   
2018-01-11   0,87   1,15   0,31  0,60%   1,960887675   
...           ...    ...    ...    ...           ...   
2023-12-21   0,34  -0,43  -0,88  2,10%    0,12872561   
2023-12-22   1,55   1,72   1,35  2,10%   0,784379335   
2023-12-26   0,51   1,12   2,05  2,10%   0,042286528   
2023-12-27  -0,06   -0,4  -0,57  2,10%  -0,616815481   
2023-12-28   0,42  -0,38  -0,38  2,10%   0,614868291   

            Variacion Logaritmica (%)  
Date                                   
2018-01-05                  -2.000065  
2018-01-08                   3.311558  
2018-01-09                  -3.817891  
2018-01-10                   1.177477  
2018-01-11                   1.

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed

           Mkt-RF    SMB    HML     RF     Fundflows  \
Date                                                   
2018-01-05   0,66  -0,35  -0,26  0,60%  -0,469042694   
2018-01-08   0,19  -0,15   0,05  0,60%   1,374904696   
2018-01-09   0,15  -0,35  -0,04  0,60%   -2,78345551   
2018-01-10  -0,07   0,07   0,57  0,60%  -0,468408524   
2018-01-11   0,87   1,15   0,31  0,60%   1,960887675   
...           ...    ...    ...    ...           ...   
2023-12-21   0,34  -0,43  -0,88  2,10%    0,12872561   
2023-12-22   1,55   1,72   1,35  2,10%   0,784379335   
2023-12-26   0,51   1,12   2,05  2,10%   0,042286528   
2023-12-27  -0,06   -0,4  -0,57  2,10%  -0,616815481   
2023-12-28   0,42  -0,38  -0,38  2,10%   0,614868291   

            Variacion Logaritmica (%)  
Date                                   
2018-01-05                   0.590977  
2018-01-08                   1.467257  
2018-01-09                  -1.207933  
2018-01-10                  -0.719792  
2018-01-11                   0.

KeyboardInterrupt: 